# Concept Completeness

In [10]:
import os

os.chdir('..')
os.getcwd()

'/media/trdp/STORAGE/3.Trabalho/PhD-Research/ThesisExperiments/2.OngoingProjects/TextualHGNN'

In [11]:
import torch
from src.models.graph_classification.gnn import DiffPool
from src.utils.general_utils import load_config
from src.models.graph_classification.train_and_evaluate import load_datasets, create_loaders
from src.models.graph_explainability.cg_evaluation_metrics import ConceptCompletenessCalculatorV2

In [12]:


LANG = "italian"
CONFIG = load_config(LANG, "src/utils/config.json")
DATASET = "Imprisonment-IT"
max_num_nodes = 1000
# MODEL_PATH = f"models/Imprisonment-IT_DiffPool_20250225_214311_lr1e-05_valmacrof1score0.6441_epoch035.pth"
MODEL_PATH = f"models/grid_search/Imprisonment-IT/Imprisonment-IT_DiffPool_20250424_213829_lr0.0001_hd100_bs4_softmaxTrue_decrease_prop0.05_valmacrof1score0.7183_epoch027.pth"
EMBEDDING_PATH = "data/external/embeddings/itwiki_20180420_100d.bin"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
# TODO: load diffpool pre-tr
diffpool_model = DiffPool(
        max_num_nodes=max_num_nodes,
        in_channels=100,
        hidden_channels=100,
        out_channels=2,
        inner_channels=16,
        softmax_assign=True,
        decrease_proportion=0.05,
    )

weights = torch.load(MODEL_PATH)
diffpool_model.load_state_dict(weights)
diffpool_model = diffpool_model.to(device=DEVICE)


/tmp/ipykernel_11285/1440289881.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights = torch.load(MODEL_PATH)


In [14]:
tgd_train, tgd_val, tgd_test = load_datasets(
    root=CONFIG["ROOT"],
    max_num_nodes=CONFIG["NUM_NODES"],
    node_feature_size=CONFIG["NODE_FEATURE_DIM"],
    lang=LANG
)

train_loader, val_loader, test_loader = create_loaders(
    tgd_train,
    tgd_val,
    tgd_test,
    batch_size=5000
)

In [15]:
completeness_calculator = ConceptCompletenessCalculatorV2(diffpool_model, train_loader, test_loader, DEVICE)

In [17]:
completeness_scores = completeness_calculator.calculate_concept_completeness()

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument mat2 in method wrapper_CUDA_mm)

In [ ]:


# Assuming `diffpool_model` is your pre-trained DiffPool model
# Assuming `train_loader` and `test_loader` are DataLoader objects for training and test datasets





print("\nFinal Concept Completeness Scores (per layer):")
for layer, (avg_score, std_score) in enumerate(completeness_scores, start=1):
    print(f"Layer {layer}: {avg_score:.4f} ± {std_score:.4f}")
